# 00 · Scraping & Pelabelan Dataset
**Mini Project NLP Part 1 · Kelompok 2**
Studi kasus: *Klasifikasi Masukan Pengguna terhadap Pengembangan Aplikasi Digital*

Notebook ini membangun dataset sendiri dari **ulasan Google Play** empat aplikasi dompet digital (e-wallet) Indonesia: **DANA, GoPay, OVO, ShopeePay**.

| Langkah | Isi |
|---|---|
| 1 | Konfigurasi (mode run otomatis, jumlah ulasan, dsb.) |
| 2 | Menentukan aplikasi sumber (package id Play Store) |
| 3 | Scraping ulasan dengan `google-play-scraper` *(atau otomatis membaca hasil scraping jika sudah ada di Input)* |
| 4 | Filter awal: buang kosong, terlalu pendek, duplikat |
| 5 | Pelabelan otomatis berbasis aturan kata kunci (5 label) |
| 6 | Cek distribusi label dan cek manual sampel per label |
| 7 | Simpan `raw_dataset.csv` dan `link_dataset.txt` |

**Lima label:** Permintaan Fitur, Bug Aplikasi, Desain Antarmuka, Performa Aplikasi, Keamanan Aplikasi.

> Ulasan Play Store **tidak punya label**. Karena itu kita membuat *pelabelan berbasis aturan* (rule-based / weak labeling): setiap ulasan diberi skor per label berdasarkan kata kunci, lalu hanya ulasan yang labelnya **jelas** yang dipakai. Ulasan yang tidak relevan atau ambigu dibuang.

## 1. Konfigurasi

Notebook ini **tidak perlu diubah kodenya** antara *Run All* dan *Save Version*. Dua hal ditentukan otomatis:

**a. Mode run** dibaca dari variabel lingkungan Kaggle `KAGGLE_KERNEL_RUN_TYPE`:

| Nilai | Kapan | Perilaku |
|---|---|---|
| `Interactive` | Run All di editor notebook | **Smoke test**: scraping hanya `SMOKE_REVIEWS_PER_STAR` ulasan per bintang, cukup untuk memastikan semua sel berjalan tanpa error |
| `Batch` | Save Version (Save & Run All) | **Full run**: scraping `REVIEWS_PER_STAR` ulasan per bintang, hasilnya menjadi dataset final |

Di luar Kaggle (lokal) variabel ini tidak ada, sehingga dianggap `Interactive`.

**b. Sumber data**: jika file `scraped_reviews_all.csv` sudah ada di `/kaggle/input` (misalnya output versi sebelumnya ditambahkan sebagai Input), file itu dibaca dan **tidak** scraping ulang. Jika tidak ada, notebook melakukan scraping (butuh *Internet ON*: Settings, Internet).

Alur yang disarankan:
1. Settings, Internet: On.
2. (Opsional) Run All untuk smoke test singkat.
3. **Save Version** (Save & Run All). Scraping penuh berjalan di versi ini.
4. Output versi tersebut (`raw_dataset.csv`, `scraped_reviews_all.csv`, `link_dataset.txt`) dijadikan Kaggle Dataset (tab Output, New Dataset) atau langsung ditambahkan sebagai Input notebook 01.

`MAX_PER_LABEL = None` artinya **tidak ada batas**: semua ulasan yang lolos pelabelan dipakai (minimal tugas 3000 baris, tidak ada batas maksimal). Isi angka (misal `3000`) hanya jika satu label terlalu dominan.

In [ ]:
import os, glob

# ---- mode run otomatis: 'Interactive' (Run All di editor) | 'Batch' (Save Version) ----
RUN_TYPE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive')
SMOKE_TEST = RUN_TYPE != 'Batch'

INPUT_DIR = '/kaggle/input'
OUTPUT_DIR = '/kaggle/working'

# ---- sumber data otomatis: pakai hasil scraping yang sudah ada di Input jika tersedia ----
EXISTING_SCRAPE = sorted(glob.glob(f'{INPUT_DIR}/**/scraped_reviews_all.csv', recursive=True))
SCRAPE = not EXISTING_SCRAPE

REVIEWS_PER_STAR = 8000        # full run: target ulasan per aplikasi untuk SETIAP bintang (1-5) -> maks 4 x 5 x 8000 = 160.000
SMOKE_REVIEWS_PER_STAR = 200   # smoke test: cukup 1 permintaan per bintang
PER_STAR = SMOKE_REVIEWS_PER_STAR if SMOKE_TEST else REVIEWS_PER_STAR

LANG, COUNTRY = 'id', 'id'
MIN_WORDS = 3             # ulasan < 3 kata dibuang (terlalu pendek untuk dilabeli, misal "mantap", "bagus sekali")
MAX_PER_LABEL = None      # None = ambil semua; isi angka jika ingin membatasi jumlah per label
RANDOM_STATE = 42

print(f'RUN_TYPE   : {RUN_TYPE} -> {"SMOKE TEST" if SMOKE_TEST else "FULL RUN"}')
print(f'Sumber data: {"scraping baru, " + str(PER_STAR) + " ulasan per bintang" if SCRAPE else "membaca " + EXISTING_SCRAPE[0]}')

## 2. Import library

`google-play-scraper` adalah library Python tidak resmi untuk mengambil data aplikasi & ulasan dari Google Play. Di Kaggle belum terpasang, jadi kita `pip install` dulu (hanya jika perlu scraping, yaitu saat `scraped_reviews_all.csv` belum ada di Input).

In [ ]:
import re, time, shutil
import pandas as pd
import matplotlib.pyplot as plt

if SCRAPE:
    !pip install -q google-play-scraper
    try:
        from google_play_scraper import app, reviews, search, Sort
    except ImportError as e:
        raise RuntimeError('google-play-scraper gagal dipasang. Aktifkan Internet (Settings, Internet: On) '
                           'atau tambahkan scraped_reviews_all.csv sebagai Input.') from e

pd.set_option('display.max_colwidth', 200)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Aplikasi sumber

Setiap aplikasi di Play Store punya **package id** unik, terlihat di URL:
`https://play.google.com/store/apps/details?id=<package_id>`

| Aplikasi | Package id |
|---|---|
| DANA | `id.dana` |
| GoPay | `com.gojek.gopay` |
| OVO | `ovo.id` |
| ShopeePay | `com.shopeepay.id` |

Fungsi `resolve_app_id` mengecek package id tersebut ke Play Store. Jika ternyata tidak ditemukan (misal package id berubah), fungsi ini otomatis **mencari** aplikasinya lewat `search()` dengan kata kunci nama aplikasi.

In [ ]:
APPS = {
    'DANA':      {'id': 'id.dana',          'query': 'DANA dompet digital'},
    'GoPay':     {'id': 'com.gojek.gopay',  'query': 'GoPay transfer payment'},
    'OVO':       {'id': 'ovo.id',           'query': 'OVO'},
    'ShopeePay': {'id': 'com.shopeepay.id', 'query': 'ShopeePay dompet serba bisa'},
}

def play_url(app_id):
    return f'https://play.google.com/store/apps/details?id={app_id}&hl={LANG}&gl={COUNTRY}'

def resolve_app_id(name, info):
    """Pastikan package id valid. Jika tidak, cari via search() dan ambil hasil yang judulnya memuat nama app."""
    try:
        meta = app(info['id'], lang=LANG, country=COUNTRY)
        print(f"[OK]  {name:<10} -> {info['id']:<18} | {meta['title']} | total ulasan: {meta.get('reviews')}")
        return info['id']
    except Exception as e:
        print(f"[!!]  {name}: package id {info['id']} gagal ({e}). Mencari ulang...")
        for hit in search(info['query'], lang=LANG, country=COUNTRY, n_hits=10):
            if hit.get('appId') and name.lower() in str(hit.get('title', '')).lower():
                print(f"[OK]  {name:<10} -> {hit['appId']} | {hit['title']}")
                return hit['appId']
        print(f'[XX]  {name} tidak ditemukan, dilewati.')
        return None

if SCRAPE:
    for name, info in APPS.items():
        info['id'] = resolve_app_id(name, info)
    APPS = {k: v for k, v in APPS.items() if v['id']}

## 4. Scraping ulasan

Strategi agar data **banyak dan beragam**:
- Ulasan diambil **per bintang (1 s.d. 5)** menggunakan `filter_score_with`. Ulasan bintang rendah banyak berisi keluhan bug/performa/keamanan, sedangkan bintang tinggi lebih sering memuji tampilan atau memberi saran fitur. Jadi semua label punya peluang muncul.
- Urutan `Sort.NEWEST` (ulasan terbaru).
- Play Store hanya memberi ±200 ulasan per permintaan, jadi kita ulang memakai `continuation_token` (seperti tombol "lihat lebih banyak") sampai target tercapai atau ulasan habis.
- `time.sleep` kecil agar tidak membebani server; `try/except` supaya satu error jaringan tidak menghentikan seluruh proses.

Hasil mentah disimpan ke `scraped_reviews_all.csv` (belum difilter, belum dilabeli).

In [ ]:
def scrape_app(name, app_id, per_star=PER_STAR, batch=200):
    rows = []
    for star in [1, 2, 3, 4, 5]:
        token, got, seen = None, 0, set()
        while got < per_star:
            try:
                result, token = reviews(app_id, lang=LANG, country=COUNTRY, sort=Sort.NEWEST,
                                        count=batch, filter_score_with=star, continuation_token=token)
            except Exception as e:
                print(f'   error {name} bintang {star}: {e} -> coba lagi 5 detik')
                time.sleep(5)
                continue
            new = [r for r in result if r['reviewId'] not in seen]
            if not new:
                break                           # ulasan untuk bintang ini sudah habis
            for r in new:
                seen.add(r['reviewId'])
                rows.append({'review_id': r['reviewId'], 'app': name, 'app_id': app_id, 'rating': r['score'],
                             'date': r['at'], 'text': r['content'], 'thumbs_up': r['thumbsUpCount']})
            got += len(new)
            if token is None or getattr(token, 'token', None) is None:
                break
            time.sleep(0.3)
        print(f'   {name:<10} bintang {star}: {got} ulasan')
    return rows

if SCRAPE:
    all_rows = []
    t0 = time.time()
    for name, info in APPS.items():
        print(f'Scraping {name} ({info["id"]}) ...')
        all_rows += scrape_app(name, info['id'])
    scraped = pd.DataFrame(all_rows)
    scraped.to_csv(f'{OUTPUT_DIR}/scraped_reviews_all.csv', index=False)
    print(f'Selesai: {len(scraped):,} ulasan dalam {(time.time()-t0)/60:.1f} menit')
else:
    scraped = pd.read_csv(EXISTING_SCRAPE[0])
    print(f'Membaca {EXISTING_SCRAPE[0]}: {len(scraped):,} ulasan (tidak scraping ulang)')
    shutil.copy(EXISTING_SCRAPE[0], f'{OUTPUT_DIR}/scraped_reviews_all.csv')   # tetap ikut di Output versi ini
    if SMOKE_TEST:   # smoke test cukup memakai sebagian data agar cepat
        scraped = scraped.sample(min(len(scraped), 5000), random_state=RANDOM_STATE)
        print(f'SMOKE TEST: memakai sampel {len(scraped):,} ulasan')

scraped.head()

In [ ]:
# Ringkasan hasil scraping per aplikasi dan per bintang
pd.crosstab(scraped['app'], scraped['rating'], margins=True, margins_name='Total')

## 5. Filter awal

1. Buang ulasan kosong.
2. Rapikan spasi berlebih / baris baru (isi teks **tidak** diubah, preprocessing lengkap ada di notebook 01).
3. Buang ulasan **< 3 kata** karena terlalu pendek untuk menunjukkan topik masukan (contoh: "mantap", "aplikasi bagus"). Ulasan 3 sampai 4 kata tetap dipakai karena banyak yang sudah jelas topiknya ("terlalu banyak iklan", "lemot banyak gangguan").
4. Buang **duplikat** (teks sama setelah huruf kecil), misalnya spam "bagus bagus bagus" yang di-copy berkali-kali.

In [ ]:
df = scraped.copy()
n0 = len(df)
df['text'] = df['text'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
df = df[df['text'].ne('') & df['text'].ne('nan')]
n1 = len(df)
df['n_words'] = df['text'].str.split().str.len()
df = df[df['n_words'] >= MIN_WORDS]
n2 = len(df)
df['_key'] = df['text'].str.lower()
df = df.drop_duplicates('_key').drop(columns='_key')
n3 = len(df)

pd.DataFrame({'tahap': ['hasil scraping', 'setelah buang kosong', f'setelah buang < {MIN_WORDS} kata', 'setelah buang duplikat'],
              'jumlah': [n0, n1, n2, n3]})

## 6. Pelabelan berbasis aturan kata kunci

### 6.1 Normalisasi khusus pelabelan
Ulasan Play Store penuh singkatan dan slang (`gak`, `gk`, `bgt`, `apk`, `lemot`, `eror`...). Agar kata kunci tetap cocok, teks **dinormalisasi sementara**:
- huruf kecil, hapus URL,
- huruf berulang dipendekkan (`lemooot` → `lemoot`),
- simbol/emoji diganti spasi,
- awalan `di` yang ditulis terpisah disambung kembali untuk kata kerja umum (`di tambahkan` → `ditambahkan`, `di gunakan` → `digunakan`),
- akhiran `-nya` dilepas dari kata minimal 6 huruf (`fiturnya` → `fitur`, `tampilannya` → `tampilan`),
- slang dipetakan ke kata baku lewat kamus `SLANG` (`gak` → `tidak`, `apk` → `aplikasi`, `gampang` → `mudah`).

Hasil normalisasi **hanya dipakai untuk mencocokkan kata kunci**. Kolom `text` yang disimpan tetap teks asli, karena preprocessing resmi (case folding, regex, stopword, stemming) dibahas di notebook 01.

In [ ]:
# ---- normalisasi ringan khusus untuk pelabelan (teks asli tidak diubah) ----
SLANG = {
    'gak': 'tidak', 'ga': 'tidak', 'gk': 'tidak', 'g': 'tidak', 'nggak': 'tidak', 'ngga': 'tidak',
    'enggak': 'tidak', 'engga': 'tidak', 'tdk': 'tidak', 'tak': 'tidak', 'kagak': 'tidak', 'kaga': 'tidak',
    'gabisa': 'tidak bisa', 'gbs': 'tidak bisa', 'gakbisa': 'tidak bisa', 'gx': 'tidak', 'ndak': 'tidak',
    'gada': 'tidak ada', 'gaada': 'tidak ada', 'kgk': 'tidak',
    'bgt': 'banget', 'bngt': 'banget', 'yg': 'yang', 'dgn': 'dengan', 'dg': 'dengan',
    'udh': 'sudah', 'udah': 'sudah', 'sdh': 'sudah', 'blm': 'belum', 'blom': 'belum',
    'apk': 'aplikasi', 'app': 'aplikasi', 'apps': 'aplikasi', 'aplikasinya': 'aplikasi', 'apl': 'aplikasi',
    'aplkasi': 'aplikasi', 'aplikasih': 'aplikasi',
    'tlg': 'tolong', 'tolonglah': 'tolong', 'mhn': 'mohon', 'krn': 'karena', 'karna': 'karena',
    'bs': 'bisa', 'bsa': 'bisa', 'dr': 'dari', 'utk': 'untuk', 'jd': 'jadi', 'jdi': 'jadi',
    'sy': 'saya', 'aku': 'saya', 'gw': 'saya', 'gue': 'saya', 'tp': 'tapi', 'tpi': 'tapi',
    'lg': 'lagi', 'lgi': 'lagi', 'sm': 'sama', 'trs': 'terus', 'trus': 'terus', 'kalo': 'kalau', 'klo': 'kalau',
    'knp': 'kenapa', 'napa': 'kenapa', 'gmn': 'bagaimana', 'gimana': 'bagaimana',
    'lemod': 'lemot', 'lelet': 'lemot', 'eror': 'error', 'erorr': 'error', 'err': 'error', 'errorr': 'error',
    'ngelag': 'lag', 'nge-lag': 'lag', 'ngeleg': 'lag', 'leg': 'lag', 'ngehang': 'hang', 'ngeheng': 'hang',
    'loding': 'loading', 'lodingnya': 'loading', 'loadingnya': 'loading',
    'tampilanya': 'tampilan', 'tampilannya': 'tampilan', 'fiturnya': 'fitur', 'fitur2': 'fitur', 'fiture': 'fitur',
    'ditambahin': 'ditambahkan', 'tambahin': 'tambahkan', 'adain': 'adakan', 'diadain': 'diadakan',
    'bikinin': 'buatkan', 'dibikinin': 'dibuatkan', 'dikasih': 'diberi',
    'pw': 'password', 'pass': 'password', 'akunnya': 'akun', 'akun2': 'akun',
    'otpnya': 'otp', 'pinnya': 'pin', 'hape': 'hp', 'warnanya': 'warna', 'tulisannya': 'tulisan',
    'hurufnya': 'huruf', 'desainnya': 'desain', 'menunya': 'menu', 'iconnya': 'icon', 'ikonnya': 'ikon',
    'tombolnya': 'tombol', 'saldonya': 'saldo', 'biaya2': 'biaya', 'tf': 'transfer', 'berfungsj': 'berfungsi', 'qiris': 'qris', 'temanya': 'tema', 'uinya': 'ui', 'darkmode': 'dark mode',
    'smg': 'semoga', 'smoga': 'semoga', 'moga': 'semoga', 'mudah2an': 'semoga', 'mudahan': 'semoga',
    'pengin': 'pengen', 'pgn': 'pengen', 'pingin': 'pengen', 'kpn': 'kapan', 'klu': 'kalau',
    'pake': 'pakai', 'dipake': 'dipakai', 'makai': 'memakai', 'digunain': 'digunakan', 'gunain': 'gunakan',
    'tambahain': 'tambahkan', 'ditambahin': 'ditambahkan', 'tambahkn': 'tambahkan', 'ditambahkn': 'ditambahkan',
    'dikasi': 'diberi', 'kasih': 'beri', 'dibuatin': 'dibuatkan', 'buatin': 'buatkan',
    'iklannya': 'iklan', 'ikln': 'iklan', 'bnyk': 'banyak', 'byk': 'banyak', 'rame': 'ramai',
    'simpel': 'simple', 'gampang': 'mudah', 'userfriendly': 'user friendly',
    'msh': 'masih', 'stak': 'stuck', 'stuk': 'stuck', 'setak': 'stuck', 'jg': 'juga', 'aja': 'saja', 'dlm': 'dalam', 'bkn': 'bukan', 'emg': 'memang',
}

# kata kerja yang sering ditulis terpisah dari awalan 'di' ("di tambahkan" -> "ditambahkan")
DI_VERBS = ('tambah|tambahkan|adakan|sediakan|buatkan|gunakan|pakai|pahami|mengerti|perbaiki|update|'
            'buka|hapus|hilangkan|kembalikan|berikan|beri|perbarui|ubah|ganti|kembangkan|hadirkan|aktifkan')
NYA_KEEP = {'bertanya', 'ditanya', 'menanya', 'tanya', 'punya', 'hanya', 'nya'}

def normalize(text: str) -> str:
    t = str(text).lower()
    t = re.sub(r'http\S+|www\.\S+', ' ', t)
    t = re.sub(r'(.)\1{2,}', r'\1\1', t)          # 'lemooot' -> 'lemoot'
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    t = re.sub(rf'\bdi (?=({DI_VERBS})\b)', 'di', t)
    t = re.sub(r'\b(tambah|ada|sedia|buat) kan\b', r'\1kan', t)   # 'tambah kan' -> 'tambahkan'
    toks = []
    for w in t.split():
        if w not in SLANG and w not in NYA_KEEP and len(w) >= 6 and w.endswith('nya'):
            w = w[:-3]                             # 'fiturnya' -> 'fitur', 'tampilannya' -> 'tampilan'
        toks.append(SLANG.get(w, w))
    return ' '.join(toks)

# contoh
contoh = 'Apknya lemooot bgt, gk bisa dibuka!!! \U0001F621 tolong di perbaiki, fiturnya kurang'
print(contoh, '\n->', normalize(contoh))

### 6.2 Kamus kata kunci per label

Setiap label punya daftar **pola regex** beserta **bobot**:
- **3** = sangat khas label tersebut (misal `lemot` → Performa, `diretas` → Keamanan, `fitur baru` → Permintaan Fitur)
- **2** = cukup khas (misal `loading`, `pin/otp`, `warna/tombol`)
- **1** = petunjuk lemah, hanya membantu jika ada petunjuk lain (misal `fitur`, `cepat`, `aman`)
- **4-5** = penanda yang hampir pasti (misal `tambahkan fitur`, `dark mode`, `dibobol`)

Penjelasan simbol regex yang dipakai:
| Simbol | Arti | Contoh |
|---|---|---|
| `\b` | batas kata, supaya `ui` tidak cocok di dalam kata `guide` | `\bui\b` |
| `( a \| b )` | salah satu alternatif | `(lemot\|lambat)` |
| `(\w+ ){0,3}` | boleh diselingi 0–3 kata apa pun | `tolong (\w+ ){0,3}ditambahkan` cocok dengan "tolong **fitur qris nya** ditambahkan" |
| `?` | bagian sebelumnya opsional | `top ?up` cocok "topup" dan "top up" |

In [ ]:
# ---- kamus kata kunci: (pola regex, bobot) ----
# bobot 3 = sangat khas label, 2 = cukup khas, 1 = petunjuk lemah (4-5 = penanda yang hampir pasti)
GAP = r'(\w+ ){0,3}'
REQ_VERB = (r'(tambahkan|ditambahkan|ditambah|menambahkan|tambah|adakan|diadakan|sediakan|disediakan|'
            r'buatkan|dibuatkan|hadirkan|dihadirkan|kembangkan|dikembangkan|menyediakan|mengadakan|menghadirkan|'
            r'membuatkan|memberikan|berikan|diberikan|diberi)')
FEATURE = r'(fitur|menu|opsi|pilihan|metode|mode|widget|integrasi|kategori|filter|notifikasi)'

KEYWORDS = {
    'Permintaan Fitur': [
        (r'\bfitur baru\b', 3),
        (rf'\b{REQ_VERB} {GAP}{FEATURE}\b', 4),
        (rf'\b{FEATURE} {GAP}(ditambahkan|ditambah|diadakan|disediakan|dibuatkan|dihadirkan|dikembangkan)\b', 4),
        (r'\b(semoga|harap|harapannya|berharap|mohon|tolong|sebaiknya|seharusnya|harusnya|alangkah baiknya|'
         r'lebih baik|kalau bisa|kalau ada|andai|coba|kapan|ingin|pengen|request|saran|mungkin|kedepannya|ke depannya|'
         r'next update|update selanjutnya|update berikutnya) ' + GAP +
         r'(ditambahkan|ditambah|tambahkan|diadakan|adakan|disediakan|sediakan|dibuatkan|buatkan|'
         r'ada fitur|ada opsi|ada pilihan|ada menu|ada mode|bisa ditambahkan|bisa diadakan)\b', 3),
        (r'\b(belum ada|tidak ada|kurang|tidak tersedia|belum tersedia|minim) (?!guna)(\w+ ){0,1}(fitur|opsi|pilihan)\b', 3),
        (r'\b(fitur|opsi|pilihan) (\w+ ){1,3}(tidak ada|belum ada|tidak tersedia|belum tersedia)\b', 2),
        (r'\b(adakan|diadakan|aktifkan|diaktifkan|kembalikan) lagi\b', 2),
        (r'\bkenapa (tidak|belum) ada (\w+ ){0,1}(fitur|opsi|pilihan|menu)\b', 2),
        (r'\bfitur ' + GAP + r'(dong|donk|plis|please|pliss|ya min|min)\b', 2),
        (r'\b(saran|usul|usulan|request|rekomendasi|masukan saya|masukan untuk|sekedar saran|sekadar saran)\b', 2),
        (r'\b(saran|usul|usulan) (saya|dari saya|saja|untuk|buat|nih|ya)\b', 2),
        (r'\b(fitur|menu|opsi) ' + GAP + r'(dihapus|dihilangkan|hilang|tidak ada lagi) ' + GAP + r'(kembalikan|dikembalikan|adakan lagi)\b', 3),
        (r'\b(kembalikan|dikembalikan) (\w+ ){0,2}(fitur|menu|opsi)\b', 3),
        (r'\bbiar bisa\b', 1), (r'\bfitur\b', 1), (r'\b(opsi|pilihan) (untuk|buat)\b', 1),
        (r'\b(semoga|harapannya) (ke depan|kedepan|kedepannya|ke depannya|kedepan nya)\b', 1),
    ],
    'Bug Aplikasi': [
        (r'\b(bug|bugs|ngebug|bugnya|nge bug)\b', 3), (r'\berror\b', 3), (r'\bforce close\b', 3), (r'\bcrash\b', 3),
        (r'\b(keluar sendiri|tertutup sendiri|close sendiri|menutup sendiri|keluar terus)\b', 3),
        (r'\bterjadi kesalahan\b', 3),
        (r'\b(tidak|belum) (bisa|dapat|mau) (dibuka|di buka|kebuka|masuk|login|log in|diakses|di akses|digunakan|dipakai)\b', 2),
        (r'\blayar (\w+ ){0,2}(putih|hitam|blank|kosong)\b', 2),
        (r'\b(tidak|belum) (berfungsi|bekerja|terbaca|terdeteksi|ke load|bisa diklik|bisa dipencet|bisa ditekan|merespon|respon|muncul|tampil)\b', 2),
        (r'\b(blank|layar putih|layar hitam|stuck|nyangkut|freeze|nge freeze)\b', 2),
        (r'\bgagal (terus|melulu|mulu|lagi )?(transaksi|transfer|bayar|pembayaran|top ?up|isi saldo|login|masuk|memuat|dimuat|kirim)\b', 2),
        (r'\b(transaksi|transfer|bayar|pembayaran|top ?up|isi saldo|login|kirim|qris) (\w+ ){0,3}(selalu |sering |terus |malah )?gagal\b', 2),
        (r'\bgagal (\w+ ){0,4}saldo (\w+ ){0,2}(terpotong|kepotong|berkurang)\b', 3),
        (r'\b(setelah|habis|sehabis|sejak) (di)?update\b', 2),
        (r'\bsaldo (sudah )?(terpotong|kepotong|berkurang) (tapi|tetapi|namun|padahal)\b', 2),
        (r'\botp (tidak|belum) (masuk|terkirim|dikirim|muncul)\b', 3),
        (r'\b(top ?up|topup|saldo|uang|duit|transferan|isi saldo) (\w+ ){0,3}(tidak|belum) (masuk|masuk masuk|terkirim|sampai)\b', 3),
        (r'\b(perbaiki|diperbaiki|benerin|dibenerin|fix)\b', 1), (r'\bgagal\b', 1),
    ],
    'Desain Antarmuka': [
        (r'\b(tampilan|desain|design|antarmuka|interface|ui|ux|layout|tata letak)\b', 3),
        (r'\b(tampilan|desain|design|ui|ux) (\w+ ){0,2}(bagus|keren|menarik|jelek|buruk|bersih|rapi|simple|baru|lama|'
         r'kuno|jadul|ramai|membingungkan|cantik|indah|elegan|nyaman|enak|ribet|sederhana|segar|fresh|kaku)\b', 2),
        (r'\b(dark mode|mode gelap|tema gelap|tema hitam|light mode|mode terang)\b', 5), (r'\b(tema|theme|skin)\b', 3),
        (r'\b(warna|font|huruf|ikon|icon|tombol|button|animasi|wallpaper|banner)\b', 2),
        (r'\b(tulisan|huruf|font|ikon|icon) (\w+ ){0,2}(kecil|besar|kekecilan|tidak terbaca|susah dibaca|buram)\b', 3),
        (r'\b(user friendly|mudah digunakan|mudah dipakai|mudah dipahami|mudah dimengerti|mudah dioperasikan|'
         r'mudah untuk digunakan|mudah penggunaan|penggunaan mudah|penggunaan (\w+ ){0,1}mudah|mudah dalam penggunaan|'
         r'simple digunakan|simple dipakai|praktis digunakan|praktis dipakai|nyaman digunakan|nyaman dipakai|intuitif)\b', 3),
        (r'\b(susah|sulit|ribet|rumit) (digunakan|dipakai|dipahami|dimengerti|dicari|ditemukan|dioperasikan)\b', 3),
        (r'\b(elegan|estetik|aesthetic|minimalis|enak dilihat|sedap dipandang|kekinian|modern)\b', 2),
        (r'\b(membingungkan|berantakan|norak|jadul|tidak rapi|terlalu ramai|ramai banget)\b', 2),
        (r'\b(kebanyakan|terlalu banyak|banyak banget|banyak sekali|penuh|full) iklan\b', 3),
        (r'\biklan (\w+ ){0,3}(mengganggu|ganggu|muncul terus|terus muncul|di mana mana|dimana mana)\b', 3),
        (r'\b(pop up|popup|iklan)\b', 1),
        (r'\b(menu|navigasi|beranda|halaman utama|halaman depan|letak)\b', 1),
        (r'\b(simple|sederhana|ribet|rumit|bingung|praktis|rapi)\b', 1),
    ],
    'Performa Aplikasi': [
        (r'\b(lemot|lemoot|lambat|lamban)\b', 3), (r'\b(lag|lagging|nge lag|patah patah|frame drop|fps)\b', 3),
        (r'\b(hang|nge hang)\b', 3),
        (r'\b(loading|memuat)\b', 2),
        (r'\b(berat|boros baterai|boros batre|makan memori|memori penuh|ukuran aplikasi|ukurannya besar|makan ram|bikin hp panas|hp panas|panas)\b', 2),
        (r'\b(server|jaringan|sistem|koneksi) (\w+ ){0,2}(down|sibuk|ramai|gangguan|bermasalah|lambat|lemot|jelek|buruk)\b', 3),
        (r'\b(gangguan|maintenance|pemeliharaan|down)\b', 2),
        (r'\b(lama banget|lama sekali|terlalu lama|kelamaan|lama loading|lama prosesnya|proses lama|prosesnya lama|lama banget prosesnya|lama sekali prosesnya)\b', 2),
        (r'\b(ringan|responsif|responsive|gesit|kencang|smooth|mulus)\b', 1), (r'\b(cepat|lancar)\b', 1),
    ],
    'Keamanan Aplikasi': [
        (r'\b(keamanan|security|privasi|privacy)\b', 3),
        (r'\b(hack|hacker|hacked|dihack|diretas|peretas|dibobol|bobol|kebobolan|dibajak|bajak|diambil alih)\b', 4),
        (r'\b(phising|phishing|social engineering|link palsu|apk palsu|cs palsu|akun palsu)\b', 4),
        (r'\b(data pribadi|kebocoran|bocor|disalahgunakan|disalah gunakan|dicuri|pencurian|diakses orang)\b', 3),
        (r'\bsaldo (saya |sy )?(hilang|raib|lenyap|terkuras|dikuras|berkurang sendiri|ludes|diambil orang)\b', 3),
        (r'\b(tanpa sepengetahuan|tanpa izin|tidak pernah (melakukan )?transaksi|bukan saya yang|login di perangkat lain|perangkat lain|perangkat tidak dikenal)\b', 3),
        (r'\b(tidak aman|kurang aman|rawan|mencurigakan)\b', 3),
        (r'\b(pin|otp|kode verifikasi|kode otp|password|kata sandi|sidik jari|fingerprint|face id|verifikasi wajah|biometrik|2fa|autentikasi)\b', 2),
        (r'\b(penipu|penipuan|ditipu|scam|scammer|modus)\b', 1),
        (r'\b(kyc|ktp|selfie)\b', 1),
        (r'\b(aman|terjamin|terpercaya)\b', 1),
    ],
    # Label bantu: topik di luar aplikasi (layanan, biaya, promo, persetujuan pinjaman, status akun).
    # Tidak pernah dipakai sebagai label; hanya bersaing skor agar ulasan seperti ini dibuang.
    'Lainnya': [
        (r'\b(cs|customer service|layanan pelanggan|admin|pelayanan|operator|respon cs|balasan cs) (\w+ ){0,3}'
         r'(lama|tidak membalas|tidak respon|tidak membantu|lambat|lemot|robot|bot|buruk|jelek)\b', 4),
        (r'\b(biaya admin|biaya tambahan|potongan|dipotong biaya|biaya layanan|biaya penanganan|potong saldo)\b', 3),
        (r'\b(promo|cashback|voucher|diskon|poin|koin|hadiah|undian)\b', 2),
        (r'\b(paylater|pinjaman|pinjam|spinjam|cicil|cicilan|kredit|limit|pengajuan|diajukan|di acc|diacc|acc|disetujui|ditolak|tagihan|denda|bunga|telat bayar|debt collector|dc)\b', 4),
        (r'\b(saran|sarankan|rekomendasi) (\w+ ){0,4}(jangan|mending|mendingan|hapus|uninstall|uninstal|pindah)\b', 4),
        (r'\b(tambah|ditambah|beri|diberi|kasih|turunkan|naikkan|naik) (\w+ ){0,1}bintang\b', 4),
        (r'\b(refund|pengembalian dana|dana kembali|uang kembali)\b', 2),
        (r'\b(upgrade|premium|gopay plus|dana premium|aktivasi|dibekukan|diblokir|dinonaktifkan)\b', 2),
        (r'\b(paket data|pulsa|kuota) (\w+ ){0,3}(tidak masuk|kurang|belum masuk)\b', 3),
        (r'\b(generate|write|create|the|you|with)\b', 2),
    ],
}
for lab, pats in KEYWORDS.items():
    print(f'{lab:<18}: {len(pats)} pola')


### 6.3 Aturan penentuan label

1. Hitung **skor** tiap label = jumlah bobot pola yang cocok.
2. Label pemenang = skor tertinggi.
3. Ulasan **dibuang** jika:
   - skor tertinggi `< MIN_SCORE (3)` → tidak cukup bukti / tidak relevan (misal "pelayanan CS lama membalas"), atau
   - selisih dengan skor kedua `< MIN_MARGIN (2)` → **ambigu** (misal ulasan yang sama kuatnya membahas bug dan lemot).

Label bantu **`Lainnya`** berisi pola topik di luar aplikasi (biaya admin, promo, persetujuan paylater atau pinjaman, status atau upgrade akun, layanan CS, paket data), ajakan ke pengguna lain ("saran saya jangan pakai"), komentar soal bintang ("nanti ditambah bintangnya"), serta teks berbahasa Inggris yang bukan ulasan. Label ini tidak pernah masuk dataset; ia hanya ikut bersaing skor sehingga ulasan yang sebenarnya membahas hal tersebut ikut dibuang (menang atau terlalu dekat skornya).

Dengan begitu hanya ulasan yang topiknya jelas yang masuk dataset, sehingga label lebih bersih untuk melatih model.

In [ ]:
LABELS = [k for k in KEYWORDS if k != 'Lainnya']   # 'Lainnya' hanya label bantu, tidak masuk dataset
COMPILED = {lab: [(re.compile(p), w) for p, w in pats] for lab, pats in KEYWORDS.items()}

MIN_SCORE = 3     # skor minimal label pemenang
MIN_MARGIN = 2    # selisih minimal dengan label kedua (kalau kurang -> ambigu, dibuang)

def score_text(text):
    t = normalize(text)
    return {lab: sum(w for rx, w in pats if rx.search(t)) for lab, pats in COMPILED.items()}

def label_text(text):
    s = score_text(text)
    ranked = sorted(s.items(), key=lambda kv: kv[1], reverse=True)
    (top, s1), (_, s2) = ranked[0], ranked[1]
    if s1 < MIN_SCORE or top == 'Lainnya':
        return None, s          # tidak relevan / tidak cukup bukti
    if s1 - s2 < MIN_MARGIN:
        return None, s          # ambigu (dua label sama kuat)
    return top, s

# contoh cara kerja skor
for t in ['Tolong tambahkan fitur transfer ke bank lain dong',
          'Aplikasi sering force close setelah update, tidak bisa dibuka',
          'Tampilan baru membingungkan, tulisannya kecil',
          'Loading lama banget, lemot parah',
          'Saldo saya hilang tanpa sepengetahuan saya, akun diretas',
          'Aplikasinya mudah digunakan, tampilannya simpel',
          'Saran saya jangan pakai aplikasi ini, mending yang lain',
          'Mantap pokoknya aplikasi terbaik']:
    lab, s = label_text(t)
    print(f'{str(lab):<18} <- {t}\n{"":<21}{s}')

### 6.4 Terapkan ke semua ulasan

In [ ]:
res = df['text'].apply(label_text)
df['label'] = res.str[0]
scores = pd.DataFrame(res.str[1].tolist(), index=df.index)
df['score_top'] = scores.max(axis=1)

ringkas = pd.Series({
    'ulasan setelah filter awal': len(df),
    'tidak relevan (skor < MIN_SCORE)': int((df['score_top'] < MIN_SCORE).sum()),
    'ambigu (selisih < MIN_MARGIN) atau topik Lainnya': int(((df['score_top'] >= MIN_SCORE) & df['label'].isna()).sum()),
    'berlabel (dipakai)': int(df['label'].notna().sum()),
})
ringkas.to_frame('jumlah')

In [ ]:
labeled = df[df['label'].notna()].copy()

# Opsional: batasi jumlah per label jika ada satu label yang terlalu dominan
if MAX_PER_LABEL is not None:
    labeled = (labeled.groupby('label', group_keys=False)
                      .apply(lambda g: g.sample(min(len(g), MAX_PER_LABEL), random_state=RANDOM_STATE)))

dist = labeled['label'].value_counts().to_frame('jumlah')
dist['persen'] = (dist['jumlah'] / dist['jumlah'].sum() * 100).round(1)
print('Total dataset berlabel:', len(labeled))
dist

## 7. Distribusi label

Dataset tidak dipaksa seimbang (tidak di-*downsample* ke label terkecil) supaya data tidak terbuang. Jika ada ketidakseimbangan, akan ditangani saat pemodelan di notebook 03 (misal `class_weight='balanced'`) dan dievaluasi dengan precision/recall/F1 per kelas, bukan hanya akurasi.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
dist['jumlah'].sort_values().plot.barh(ax=axes[0], color='#4C72B0')
axes[0].set_title('Jumlah ulasan per label')
for i, v in enumerate(dist['jumlah'].sort_values()):
    axes[0].text(v, i, f' {v}', va='center')
pd.crosstab(labeled['label'], labeled['app']).plot.barh(ax=axes[1], stacked=True)
axes[1].set_title('Komposisi aplikasi per label')
plt.tight_layout(); plt.show()

pd.crosstab(labeled['label'], labeled['app'], margins=True, margins_name='Total')

In [ ]:
# Sebaran rating (bintang) per label: masuk akal jika Bug/Performa/Keamanan dominan bintang rendah
pd.crosstab(labeled['label'], labeled['rating'], normalize='index').mul(100).round(1)

## 8. Cek manual sampel per label

Karena label dibuat otomatis, kita **wajib memeriksa** apakah hasilnya masuk akal. Sel di bawah menampilkan 10 ulasan acak per label. Baca dan nilai: apakah labelnya benar?

Jika ada pola salah label yang sering muncul, perbaiki kamus `KEYWORDS` (tambah/kurangi pola atau ubah bobot) lalu jalankan ulang dari bagian 6. Ganti `random_state` untuk melihat sampel lain.

In [ ]:
for lab in LABELS:
    sub = labeled[labeled['label'] == lab]
    print(f'\n=========== {lab} ({len(sub)} ulasan) ===========')
    for _, r in sub.sample(min(10, len(sub)), random_state=RANDOM_STATE).iterrows():
        print(f'[{r.app} | {r.rating}*] {r.text[:220]}')

In [ ]:
# Contoh ulasan yang DIBUANG karena ambigu (skor dua label sama kuat) -> untuk penjelasan saat presentasi
ambigu = df[(df['score_top'] >= MIN_SCORE) & df['label'].isna()]
for t in ambigu['text'].sample(min(5, len(ambigu)), random_state=RANDOM_STATE):
    print({k: v for k, v in score_text(t).items() if v}, '<-', t[:200])

## 9. Simpan output

- **`raw_dataset.csv`**: kolom wajib `text, label` + kolom tambahan `app, rating, date` (berguna untuk EDA). Teks masih asli (belum dibersihkan) → dibersihkan di notebook 01 menjadi `clean_dataset_part1.csv`.
- **`link_dataset.txt`**: daftar URL halaman Play Store sumber data.
- **`scraped_reviews_all.csv`**: semua ulasan mentah sebelum filter (hasil scraping baru, atau salinan dari Input). Jika output versi ini ditambahkan sebagai Input, run berikutnya membaca file ini dan tidak scraping ulang.

In [ ]:
out = labeled[['text', 'label', 'app', 'rating', 'date']].sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
out.to_csv(f'{OUTPUT_DIR}/raw_dataset.csv', index=False)

APP_IDS = {'DANA': 'id.dana', 'GoPay': 'com.gojek.gopay', 'OVO': 'ovo.id', 'ShopeePay': 'com.shopeepay.id'}
if 'app_id' in scraped.columns:   # pakai package id yang benar-benar dipakai saat scraping
    APP_IDS.update(scraped.groupby('app')['app_id'].first().to_dict())
lines = ['Sumber dataset: ulasan pengguna Google Play Store (scraping dengan library google-play-scraper, bahasa id, negara id)', '']
lines += [f'{name}: {play_url(app_id)}' for name, app_id in APP_IDS.items()]
with open(f'{OUTPUT_DIR}/link_dataset.txt', 'w') as f:
    f.write('\n'.join(lines) + '\n')

print(open(f'{OUTPUT_DIR}/link_dataset.txt').read())
print('raw_dataset.csv :', out.shape)
out.head(10)

**Selesai.** Lanjut ke `01_eda_preprocessing.ipynb` yang membaca `raw_dataset.csv`.